# RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [ ]:
!pip install langchain langchain-text-splitters     

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader , PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\kotha\AppData\Local\Temp\ipykernel_20384\4091801365.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader , PyMuPDFLoader
e:\RAG\rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# DATA INGESTION

### Read all pdf inside the directory

- this function is from inside a folder, it reads all the pdf files , it reads all the content, add the meta data info, and storing in the variable all_documents

In [3]:
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"   Loaded {len(documents)} pages")
        
        except Exception as e:
            print(f" Error: {e}")
    
    print(f" \n Total documents loaded:  {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data/pdf_files")    # function calling


Found 3 PDF files to process

Processing: NLP_Handbook_TG_CPGET_2026.pdf
   Loaded 17 pages

Processing: NoSQL_Databases_Handbook_CPGET2026.pdf
   Loaded 13 pages

Processing: TG_CPGET_2026_Machine_Learning_Handbook.pdf
   Loaded 17 pages
 
 Total documents loaded:  47


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m149', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36', 'creationdate': '2026-07-04T08:49:34+00:00', 'title': 'NLP_Handbook_TG_CPGET_2026.md', 'moddate': '2026-07-04T08:49:34+00:00', 'source': '..\\data\\pdf_files\\NLP_Handbook_TG_CPGET_2026.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1', 'source_file': 'NLP_Handbook_TG_CPGET_2026.pdf', 'file_type': 'pdf'}, page_content='Natural Language Processing — Revision Handbook\nTG CPGET 2026 Exam Preparation\nThis handbook covers all 8 topics from your syllabus: Language Processing & Python, Text\nCorpora & Lexical Resources, Processing Raw Text, Categorizing & Tagging Words,\nLearning to Classify Text, Deep Learning for NLP, Extracting Information from Text, and\nAnalysing Sentence Structure.\nEach section has: Concept (simple language) → Key Deﬁnitions/Formulas → Worked\nExamples → Practice Questions.\nTABLE OF CONTENT

# CHUNKING

## Text splitting get into chunks

- chunk_size → Maximum size of each chunk (default 1000 characters).
- chunk_overlap → Number of characters shared between consecutive chunks (default 200). - the important context isn't lost.
- RecursiveCharacterTextSplitter measures the text using characters. chunk_size=1000 means 1000 characters(a,b,c,d ,.....)
- The splitter doesn't randomly cut text. It first tries to split at sensible boundaries. Order matters. - Split by paragraph. - If paragraph is too big, Split by line. - If line is too big, Split by spaces. - Last option split by Character.

#### Remember every chunk is still a Document.

- The splitter tries increasingly finer separators until the chunk fits within chunk_size, It recursively falls back to the next separator, which is why it's called RecursiveCharacterTextSplitter.

#### RecursiveCharacterTextSplitter splits large Document objects into smaller chunks while preserving context using overlap. It tries to split at natural boundaries such as paragraphs, then lines, then spaces, and finally individual characters. These smaller chunks are easier to embed, store in a vector database, and retrieve accurately during RAG.

In [5]:
def split_documents(documents, chunk_size = 1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [6]:
chunks = split_documents(all_pdf_documents)
chunks

Split 47 documents into 124 chunks

Example chunk:
Content: Natural Language Processing — Revision Handbook
TG CPGET 2026 Exam Preparation
This handbook covers all 8 topics from your syllabus: Language Processing & Python, Text
Corpora & Lexical Resources, Pro...
Metadata: {'producer': 'Skia/PDF m149', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36', 'creationdate': '2026-07-04T08:49:34+00:00', 'title': 'NLP_Handbook_TG_CPGET_2026.md', 'moddate': '2026-07-04T08:49:34+00:00', 'source': '..\\data\\pdf_files\\NLP_Handbook_TG_CPGET_2026.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1', 'source_file': 'NLP_Handbook_TG_CPGET_2026.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m149', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36', 'creationdate': '2026-07-04T08:49:34+00:00', 'title': 'NLP_Handbook_TG_CPGET_2026.md', 'moddate': '2026-07-04T08:49:34+00:00', 'source': '..\\data\\pdf_files\\NLP_Handbook_TG_CPGET_2026.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1', 'source_file': 'NLP_Handbook_TG_CPGET_2026.pdf', 'file_type': 'pdf'}, page_content='Natural Language Processing — Revision Handbook\nTG CPGET 2026 Exam Preparation\nThis handbook covers all 8 topics from your syllabus: Language Processing & Python, Text\nCorpora & Lexical Resources, Processing Raw Text, Categorizing & Tagging Words,\nLearning to Classify Text, Deep Learning for NLP, Extracting Information from Text, and\nAnalysing Sentence Structure.\nEach section has: Concept (simple language) → Key Deﬁnitions/Formulas → Worked\nExamples → Practice Questions.\nTABLE OF CONTENT

# EMBEDDING

### Text -> Vectors

In [ ]:
!pip install sentence-transformers faiss-cpu chromadb

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer 
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()       

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2038.58it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\kotha\AppData\Local\Temp\ipykernel_20384\829750999.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


- This function runs automatically when you create the object.
                - Remember which embedding model to use.
                - Create an empty place to store the model.
                - Load the model immediately.

- Its only job is to load the embedding model.

- Take text and convert it into vectors (embeddings). - Text -> Embedding Model -> Vectors

    """
        Create EmbeddingManager
                ↓
        Run __init__()
                ↓
        Load model
                ↓
        Ready
    """

##### This EmbeddingManager class is a wrapper around the SentenceTransformer model that simplifies embedding generation in a RAG pipeline. 
##### When an object is created, it automatically loads the specified embedding model from Hugging Face and stores it in memory. 
##### The generate_embeddings() method then takes a list of text chunks, converts each one into a fixed-length numerical vector (384 dimensions for all-MiniLM-L6-v2), and returns them as a NumPy array(x,384). 
##### These embeddings are the semantic representations that will later be stored in a vector database like FAISS or ChromaDB for efficient similarity search during retrieval.

# VECTOR STORE

In [10]:
class VectorStore():
    def __init__(self, collection_name: str = "pdf_documents", persist_directory : str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok = True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "PDF document embeddings for RAG",
                            "hnsw:space": "cosine",
                            "embedding_manager": embedding_manager.model_name})
            print(f"Vector store initialized. Collection: {self.collection_name}, Persist directory: {self.persist_directory}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vactor store: {e}")
            raise

    def add_documents(self, documents: List[any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store ...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise 

vectorstore = VectorStore()
vectorstore


Vector store initialized. Collection: pdf_documents, Persist directory: ../data/vector_store
Existing documents in collection: 0


__init__()

- Runs automatically.
- Stores collection name.
- Stores folder path.
- Creates empty client.
- Creates empty collection.
- Calls _initialize_store().

_initialize_store()

- Create/Open the ChromaDB database.
- Create folder if it doesn't exist.
- Create a ChromaDB client.
- Get or create a collection with the given name and metadata. Args: name: The name of the collection to get or create metadata: Optional metadata to associate with the collection. If the collection already exists, the metadata provided is ignored. If the collection does not exist, the new collection will be created with the provided metadata. 
- Creates folder.
- Opens ChromaDB.
- Opens/creates collection.
- Prints status.
- Counts documents.

add_documents()

- Store documents + embeddings inside ChromaDB.
- stores ids, stores meta data, stores actual chunk text, stores vectors
- Checks document/embedding count.
- Creates IDs.
- Copies metadata.
- Adds extra metadata.
- Stores text.
- Converts embeddings to lists.
- Saves everything into ChromaDB. 

embedding.tolist() converts NumPy Array to python list ,, it becomes list of lists
- This changes: array([0.12, 0.45, ...])
- into: [0.12, 0.45, ...]

self.collection.add(....) ==> It inserts data into the vector database.
- ids = unique ids -> doc_a12
- embeddungs = vectors are stored
- metadatas = stored along side the vectors
- documents = original chunk text is also stored


| ID    | Embedding            | Document                     | Metadata     |
| ----- | -------------------- | ---------------------------- | ------------ |
| doc_1 | `[0.12, -0.45, ...]` | "RAG combines retrieval..."  | `{"page":1}` |
| doc_2 | `[0.67, 0.91, ...]`  | "Embeddings convert text..." | `{"page":2}` |


In [11]:
chunks

[Document(metadata={'producer': 'Skia/PDF m149', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36', 'creationdate': '2026-07-04T08:49:34+00:00', 'title': 'NLP_Handbook_TG_CPGET_2026.md', 'moddate': '2026-07-04T08:49:34+00:00', 'source': '..\\data\\pdf_files\\NLP_Handbook_TG_CPGET_2026.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1', 'source_file': 'NLP_Handbook_TG_CPGET_2026.pdf', 'file_type': 'pdf'}, page_content='Natural Language Processing — Revision Handbook\nTG CPGET 2026 Exam Preparation\nThis handbook covers all 8 topics from your syllabus: Language Processing & Python, Text\nCorpora & Lexical Resources, Processing Raw Text, Categorizing & Tagging Words,\nLearning to Classify Text, Deep Learning for NLP, Extracting Information from Text, and\nAnalysing Sentence Structure.\nEach section has: Concept (simple language) → Key Deﬁnitions/Formulas → Worked\nExamples → Practice Questions.\nTABLE OF CONTENT

In [12]:
# Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

# Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# Store in the vector database
vectorstore.add_documents(chunks, embeddings)


Generating embeddings for 124 texts...


Batches: 100%|██████████| 4/4 [00:12<00:00,  3.23s/it]


Generated embeddings with shape: (124, 384)
Adding 124 documents to vector store ...
Successfully added 124 documents to vector store
Total documents in collection: 124


In [13]:
data = vectorstore.collection.get()

print(data.keys())

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])


In [14]:
print(data["ids"][:5])

['doc_b694bf96_0', 'doc_2fac76bf_1', 'doc_35288bb2_2', 'doc_9a9eb29f_3', 'doc_a0093a8d_4']


In [15]:
import pandas as pd

data = vectorstore.collection.get()

df = pd.DataFrame({
    "ID": data["ids"],
    "Document": data["documents"],
    "Metadata": data["metadatas"]
})

df.head()

,ID,Document,Metadata
0,doc_b694bf96_0,Natural Language Processing — Revision Handboo...,"{'total_pages': 17, 'creator': 'Mozilla/5.0 (W..."
1,doc_2fac76bf_1,10. Mixed Mock Test (10 Questions)\n1. Languag...,"{'moddate': '2026-07-04T08:49:34+00:00', 'page..."
2,doc_35288bb2_2,"Dictionary – key-value pairs, used for frequen...","{'producer': 'Skia/PDF m149', 'title': 'NLP_Ha..."
3,doc_9a9eb29f_3,Key Formula:\n(Value close to 1 → highly diver...,"{'page': 1, 'creationdate': '2026-07-04T08:49:..."
4,doc_a0093a8d_4,Example 4:nltk.Text.concordance('love') on a n...,"{'content_length': 920, 'page': 2, 'creationda..."


In [16]:
for key, value in data.items():
    if value is None:
        print(f"{key}: None")
    else:
        print(f"{key}: {len(value)}")

ids: 124
embeddings: None
documents: 124
uris: None
included: 2
data: None
metadatas: 124


# RETRIEVER PIPELINE FROM VECTORE STORE

searches the vector database to find the most relevant chunks.

In [36]:
class RAGRetriever:
    def __init__(self, vector_store : VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str,Any]]:
        print(f"Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}")
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try: 
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )
            print("----------------")
            print(results["distances"])
            print(results["documents"])

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata,distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance

                    print("----------------")
                    # print(document[:100])
                    print("Distance:", distance)
                    print("Similarity:", similarity_score)

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            'distance' : distance,
                            'rank' : i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents after filtering")
            else:
                print("No documents retrieved from the vector store (Not Found)")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever = RAGRetriever(vectorstore, embedding_manager)
rag_retriever

In [37]:
rag_retriever.retrieve("Language Processing and Python", top_k=3, score_threshold=0.5)

Retrieving documents for query: 'Language Processing and Python' with top_k=3 and score_threshold=0.5
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.58it/s]

Generated embeddings with shape: (1, 384)
----------------
[[0.33847975730895996, 0.34174013137817383, 0.5033931136131287]]
[['10. Mixed Mock Test (10 Questions)\n1. Language Processing and Python\n1.1 Concept in Simple Language\nNLP is the branch of AI concerned with making computers understand, interpret, and\ngenerate human language. Python is the preferred language for NLP because of libraries\nlike NLTK (Natural Language Toolkit), which gives ready-made tools for tokenizing,\ntagging, parsing, and analysing text.\nThink of text as raw, messy material (like cotton) and NLP as the spinning mill that turns it\ninto usable thread (structured data) — tokens, tags, trees, meanings.\nCore built-in Python text structures for NLP:\nString – sequence of characters ("Hyderabad" )\nList – ordered, mutable collection ([\'NLP\',\'is\',\'fun\'] )\nTuple – ordered, immutable collection', 'Natural Language Processing — Revision Handbook\nTG CPGET 2026 Exam Preparation\nThis handbook covers all 8 t

[{'id': 'doc_2fac76bf_1',
  'content': '10. Mixed Mock Test (10 Questions)\n1. Language Processing and Python\n1.1 Concept in Simple Language\nNLP is the branch of AI concerned with making computers understand, interpret, and\ngenerate human language. Python is the preferred language for NLP because of libraries\nlike NLTK (Natural Language Toolkit), which gives ready-made tools for tokenizing,\ntagging, parsing, and analysing text.\nThink of text as raw, messy material (like cotton) and NLP as the spinning mill that turns it\ninto usable thread (structured data) — tokens, tags, trees, meanings.\nCore built-in Python text structures for NLP:\nString – sequence of characters ("Hyderabad" )\nList – ordered, mutable collection ([\'NLP\',\'is\',\'fun\'] )\nTuple – ordered, immutable collection',
  'metadata': {'source': '..\\data\\pdf_files\\NLP_Handbook_TG_CPGET_2026.pdf',
   'page': 0,
   'file_type': 'pdf',
   'creationdate': '2026-07-04T08:49:34+00:00',
   'source_file': 'NLP_Handbook_

__init__()
- Stores the vector database.
- Stores the embedding model.
- Makes them available to other functions.

retrieve()
- Accepts the user's question.
- Converts the question into an embedding.
- Searches the vector database.
- Retrieves the top k closest chunks.
- Computes similarity scores.
- Filters low-quality matches.
- Returns clean, structured retrieval results.

In [47]:
print(vectorstore.collection.metadata)
print(vectorstore.collection.count())

print(vectorstore.persist_directory)

# Delete the entire collection
# vectorstore.client.delete_collection(vectorstore.collection_name)

{'description': 'PDF document embeddings for RAG', 'hnsw:space': 'cosine', 'embedding_manager': 'all-MiniLM-L6-v2'}
124
../data/vector_store


In [41]:
print(vectorstore.persist_directory)

../data/vector_store
